In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.config("spark.dynamicAllocation.enabled", "false") \
.config("spark.shulffle.service.enabled", "false") \
.config("spark.executor.instnaces", "3") \
.enableHiveSupport() \
.master("yarn") \
.appName("caching_external") \
.getOrCreate()

### In the dataframe section (dataframe_9_external_table.ipynb) we created a temporary view of the table that was read using spark.read and then used this temporary view to create and external  table and gave external location by ""LOCATION '/user/itv025320/external_tables/order' clause in the CTAS like as:
```
spark.sql("""
    CREATE TABLE big_data_025320.order_external
    USING PARQUET
    LOCATION '/user/itv025320/external_tables/order'
    AS SELECT 
        order_id,
        CAST(order_date AS TIMESTAMP) as order_date,
        customer_id,
        order_status
    FROM order_view
""")
```

### This time around we're going to use the same external location for **external table** where we created the table by simply **pointing to our previous location.** So no CTAS as the data/table already exists in external location. Just provide it with the metadata and location to point to external table.

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS caching_250320")

""


### It is worth noting that the **external table in parquet format** and has schema as:
```
+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |int                                                               |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |int                                                               |null   |
|order_status                |string                                                            |null   |
```

### Scenario 1: using CSV instead of parquet with different schema
```
|order_id        long
|order_date      timestamp
|customer_id     long
|order_status    string
```

In [6]:
spark.sql("""
    CREATE TABLE caching_250320.order_external_caching
    (order_id long, order_date timestamp, customer_id long, order_status string)
    USING CSV
    LOCATION "/user/itv025320/external_tables/order"
""")

""


In [7]:
! hdfs dfs -ls -h external_tables/order

Found 2 items
-rw-r--r--   3 itv025320 supergroup          0 2026-03-30 05:41 external_tables/order/_SUCCESS
-rw-r--r--   3 itv025320 supergroup    476.1 K 2026-03-30 05:41 external_tables/order/part-00000-527264b6-6f59-4738-8ca9-853669a79483-c000.snappy.parquet


In [8]:
spark.catalog.isCached("caching_250320.order_external_caching")

False

In [9]:
spark.sql("select * from caching_250320.order_external_caching").show()

+--------+----------+-----------+------------+
|order_id|order_date|customer_id|order_status|
+--------+----------+-----------+------------+
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|   

In [10]:
spark.sql("describe extended caching_250320.order_external_caching").show(truncate=False)

+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |bigint                                                            |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |bigint                                                            |null   |
|order_status                |string                                                            |null   |
|                            |                                                                  |       |
|# Detailed Table Information|                                                                  |       |
|Database                    |caching_250320  

In [11]:
spark.sql("DROP TABLE caching_250320.order_external_caching")

""


### Scenario 2: using CSV with same schema
```
|order_id        int
|order_date      timestamp
|customer_id     int
|order_status    string
```

In [12]:
spark.sql("""
    CREATE TABLE caching_250320.order_external_caching
    (order_id int, order_date timestamp, customer_id int, order_status string)
    USING CSV
    LOCATION "/user/itv025320/external_tables/order"
""")

""


In [13]:
spark.sql("select * from caching_250320.order_external_caching").show()

+--------+----------+-----------+------------+
|order_id|order_date|customer_id|order_status|
+--------+----------+-----------+------------+
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|      null|       null|        null|
|    null|   

In [15]:
spark.sql("describe extended caching_250320.order_external_caching").show(truncate=False)

+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |int                                                               |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |int                                                               |null   |
|order_status                |string                                                            |null   |
|                            |                                                                  |       |
|# Detailed Table Information|                                                                  |       |
|Database                    |caching_250320  

In [16]:
spark.sql("DROP TABLE caching_250320.order_external_caching")

""


### Scenario 3: using parquet with different schema
```
|order_id        long
|order_date      timestamp
|customer_id     long
|order_status    string
```

In [17]:
spark.sql("""
    CREATE TABLE caching_250320.order_external_caching
    (order_id long, order_date timestamp, customer_id long, order_status string)
    USING PARQUET
    LOCATION "/user/itv025320/external_tables/order"
""")

""


In [19]:
spark.sql("describe extended caching_250320.order_external_caching").show(truncate=False)

+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |bigint                                                            |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |bigint                                                            |null   |
|order_status                |string                                                            |null   |
|                            |                                                                  |       |
|# Detailed Table Information|                                                                  |       |
|Database                    |caching_250320  

In [20]:
spark.sql("select * from caching_250320.order_external_caching").show()

Py4JJavaError: An error occurred while calling o196.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 2.0 failed 4 times, most recent failure: Lost task 0.3 in stage 2.0 (TID 5) (w03.itversity.com executor 2): org.apache.spark.sql.execution.QueryExecutionException: Parquet column cannot be converted in file hdfs://m01.itversity.com:9000/user/itv025320/external_tables/order/part-00000-527264b6-6f59-4738-8ca9-853669a79483-c000.snappy.parquet. Column: [order_id], Expected: bigint, Found: INT32
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:179)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:93)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:503)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spar

#### org.apache.spark.sql.execution.QueryExecutionException: Parquet column cannot be converted **Column: [order_id], Expected: bigint, Found: INT32**
#### **SchemaColumnConvertNotSupportedException**

In [21]:
spark.sql("DROP TABLE caching_250320.order_external_caching")

""


### Scenario 4: using parquet with correct schema

```
|order_id        int
|order_date      timestamp
|customer_id     int
|order_status    string
```

In [22]:
spark.sql("""
    CREATE TABLE caching_250320.order_external_caching
    (order_id int, order_date timestamp, customer_id int, order_status string)
    USING PARQUET
    LOCATION "/user/itv025320/external_tables/order"
""")

""


In [25]:
spark.sql("DESCRIBE EXTENDED caching_250320.order_external_caching").show(truncate=False)

+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |int                                                               |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |int                                                               |null   |
|order_status                |string                                                            |null   |
|                            |                                                                  |       |
|# Detailed Table Information|                                                                  |       |
|Database                    |caching_250320  

In [26]:
read_df = spark.sql("select * from caching_250320.order_external_caching")

In [27]:
read_df.show()

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
|       6|2013-07-25 00:00:00|       7130|       COMPLETE|
|       7|2013-07-25 00:00:00|       4530|       COMPLETE|
|       8|2013-07-25 00:00:00|       2911|     PROCESSING|
|       9|2013-07-25 00:00:00|       5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:00|       5648|PENDING_PAYMENT|
|      11|2013-07-25 00:00:00|        918| PAYMENT_REVIEW|
|      12|2013-07-25 00:00:00|       1837|         CLOSED|
|      13|2013-07-25 00:00:00|       9149|PENDING_PAYMENT|
|      14|2013-07-25 00:00:00|       9842|     PROCESSIN